# Evaluation 05 — Box plots MAE/MSE

Compara modelos a partir das previsões em `previsoes/...`. A coluna `step` identifica o ponto temporal previsto. Como o rolling forecast tem sobreposição, aplica `PARADIGMA = 'first' | 'median' | 'last'` por `(modelo, papel, step)`, busca o dado real, calcula erro, MAE e MSE, e plota por grupo de papel e agregado.

In [ ]:
from pathlib import Path
import os, re, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

# Paradigma de avaliação da janela deslizante:
#   "first"  -> primeira previsão de cada ponto
#   "median" -> mediana das previsões de cada ponto
#   "last"   -> última previsão de cada ponto
PARADIGMA = "first"
assert PARADIGMA in {"first", "median", "last"}

# Métrica de avaliação:
# MAE e MSE por lote/janela e série, quando há coluna de lote;
# caso contrário, MAE e MSE por série + agregado médio entre séries.
METRICAS = ["MAE", "MSE"]

CONFIG = {
    "col_step_previsao": "step",
    "col_lote_previsao": None,  # ou: "janela", "lote", "window", "origin"...
    "candidatos_col_lote_previsao": ["janela", "janela_id", "lote", "lote_id", "window", "window_id", "batch", "batch_id", "origin", "cutoff", "rolling_origin", "epoch"],
    "col_step_real": "date",
    "col_valor_real": "data",
    "col_papel_real": "cols",
    "pastas_dados_reais": ["../data", "./data", "../datasets", "./datasets", ".", "/mnt/data"],
    "arquivos_dados_reais": {
        "b3_daily_financeiro": "b3_daily_financeiro.csv",
        "b3_daily_financeiro_ohlcv": "b3_daily_financeiro_ohlcv.csv",
    },
    "padroes_previsao": ["*.parquet", "*.csv", "*.feather", "*.pkl", "*.pickle"],
    "ignorar_arquivos_contendo": ["train_loss", "loss", "history", "metric", "metrics", "resumo", "summary", "config", "checkpoint", "model", "scaler", "normalizer"],
}

MAPA_GRUPO_PAPEL = {"3": "ON_3", "4": "PN_4", "5": "PNA_5", "6": "PNB_6", "11": "UNIT_11"}
PALETA_PADRAO = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# Edite este vetor. O dataset é inferido de ../previsoes/<dataset>/<modelo>/<data>.
# Se necessário, informe "dataset" ou "dados_reais" explicitamente.
registra_modelo = [
    # {"nome": "Modelo_1", "endereco": "../previsoes/b3_daily_financeiro/Modelo_1/2026-06-01", "cor": "#1f77b4"},
    # {"nome": "Modelo_2", "endereco": "../previsoes/b3_daily_financeiro/Modelo_2/2026-06-01", "cor": "#ff7f0e"},
]
for i, r in enumerate(registra_modelo):
    r.setdefault("cor", PALETA_PADRAO[i % len(PALETA_PADRAO)])

In [ ]:
def ler_arquivo(path):
    path = Path(path)
    if path.suffix.lower() == ".csv": return pd.read_csv(path)
    if path.suffix.lower() == ".parquet": return pd.read_parquet(path)
    if path.suffix.lower() == ".feather": return pd.read_feather(path)
    if path.suffix.lower() in [".pkl", ".pickle"]: return pd.read_pickle(path)
    raise ValueError(f"Extensão não suportada: {path}")

def inferir_dataset(endereco):
    partes = Path(str(endereco)).parts
    if "previsoes" in partes and partes.index("previsoes") + 1 < len(partes):
        return partes[partes.index("previsoes") + 1]
    return None

def localizar_real(reg, cfg):
    if reg.get("dados_reais"):
        p = Path(reg["dados_reais"]).expanduser()
        if p.exists(): return p
        raise FileNotFoundError(p)
    dataset = reg.get("dataset") or inferir_dataset(reg["endereco"])
    if not dataset:
        raise ValueError("Informe 'dataset' ou 'dados_reais' em registra_modelo.")
    nomes = []
    if dataset in cfg["arquivos_dados_reais"]: nomes.append(cfg["arquivos_dados_reais"][dataset])
    nomes += [f"{dataset}.csv", f"{dataset}.parquet", f"{dataset}.feather"]
    for pasta in cfg["pastas_dados_reais"]:
        for nome in nomes:
            p = Path(pasta).expanduser() / nome
            if p.exists(): return p
    raise FileNotFoundError(f"Dados reais não encontrados para dataset={dataset}.")

def carregar_real(reg, cfg):
    p = localizar_real(reg, cfg)
    df = ler_arquivo(p).copy()
    cs, cv, cp = cfg["col_step_real"], cfg["col_valor_real"], cfg["col_papel_real"]
    if {cs, cv, cp}.issubset(df.columns):
        out = df[[cs, cv, cp]].rename(columns={cs:"step", cv:"y_true", cp:"papel"})
    else:
        step_col = next((c for c in [cs, "step", "date", "tempo", "timestamp"] if c in df.columns), None)
        if step_col is None: raise ValueError(f"Coluna temporal não encontrada em {p}")
        vals = [c for c in df.columns if c != step_col and pd.api.types.is_numeric_dtype(df[c])]
        out = df.melt(id_vars=[step_col], value_vars=vals, var_name="papel", value_name="y_true").rename(columns={step_col:"step"})
    out["papel"] = out["papel"].astype(str)
    out["step"] = pd.to_numeric(out["step"], errors="coerce")
    out["y_true"] = pd.to_numeric(out["y_true"], errors="coerce")
    out = out.dropna(subset=["step", "papel", "y_true"])
    out["step"] = out["step"].astype(int)
    return out.drop_duplicates(["papel", "step"], keep="last")

def arquivos_previsao(endereco, cfg):
    p = Path(endereco).expanduser()
    if p.is_file(): return [p]
    if not p.exists(): raise FileNotFoundError(p)
    arqs = []
    for padrao in cfg["padroes_previsao"]: arqs += list(p.rglob(padrao))
    ign = [x.lower() for x in cfg["ignorar_arquivos_contendo"]]
    return sorted({a for a in arqs if not any(x in a.name.lower() for x in ign)})

def grupo_papel(papel):
    m = re.search(r"(\d+)$", str(papel))
    return MAPA_GRUPO_PAPEL.get(m.group(1), f"TIPO_{m.group(1)}") if m else "sem_grupo"

def carregar_previsoes(reg, real, cfg):
    tickers = set(real["papel"].astype(str).unique())
    partes, ignorados = [], []
    for ordem_arq, arq in enumerate(arquivos_previsao(reg["endereco"], cfg)):
        try: df = ler_arquivo(arq).copy()
        except Exception as e:
            ignorados.append((arq, f"erro: {e}")); continue
        if cfg["col_step_previsao"] not in df.columns:
            ignorados.append((arq, "sem step")); continue
        ticker_cols = [c for c in df.columns if str(c) in tickers]
        if not ticker_cols:
            ignorados.append((arq, "sem colunas de papéis dos dados reais")); continue
        df = df.dropna(subset=ticker_cols, how="all")
        if df.empty: continue
        df["_arquivo_origem"], df["_ordem_arquivo"], df["_linha_no_arquivo"] = str(arq), ordem_arq, np.arange(len(df))
        long = df.melt(id_vars=[c for c in df.columns if c not in ticker_cols], value_vars=ticker_cols, var_name="papel", value_name="y_pred")
        long = long.rename(columns={cfg["col_step_previsao"]:"step"})
        long["modelo"] = reg["nome"]
        long["step"] = pd.to_numeric(long["step"], errors="coerce")
        long["y_pred"] = pd.to_numeric(long["y_pred"], errors="coerce")
        long = long.dropna(subset=["step", "papel", "y_pred"])
        long["step"] = long["step"].astype(int)
        long["ordem_previsao"] = long["_ordem_arquivo"].astype("int64") * 10**12 + long["_linha_no_arquivo"].astype("int64")
        partes.append(long)
    if ignorados:
        print(f"Arquivos ignorados para {reg['nome']}:")
        for a,m in ignorados[:15]: print(f"  - {a}: {m}")
    if not partes: raise RuntimeError(f"Nenhum arquivo de previsão válido para {reg['nome']}.")
    return pd.concat(partes, ignore_index=True)

def aplicar_paradigma(pred, paradigma):
    keys = ["modelo", "papel", "step"]
    if paradigma == "median":
        return pred.groupby(keys, as_index=False).agg(y_pred=("y_pred", "median"), n_previsoes=("y_pred", "count"), primeira_ordem=("ordem_previsao", "min"), ultima_ordem=("ordem_previsao", "max"))
    counts = pred.groupby(keys, as_index=False).size().rename(columns={"size":"n_previsoes"})
    asc = paradigma == "first"
    return pred.sort_values(keys + ["ordem_previsao"], ascending=[True, True, True, asc]).groupby(keys, as_index=False).first().merge(counts, on=keys, how="left")

def construir_base(registros, cfg, paradigma):
    if not registros: raise ValueError("Preencha registra_modelo antes de executar.")
    bases, brutas = [], []
    for reg in registros:
        print(f"\nCarregando modelo: {reg['nome']}")
        real = carregar_real(reg, cfg)
        raw = carregar_previsoes(reg, real, cfg)
        pred = aplicar_paradigma(raw, paradigma)
        base = pred.merge(real, on=["papel", "step"], how="inner")
        if base.empty: raise RuntimeError(f"Merge previsão x real vazio para {reg['nome']}.")
        base["grupo_papel"] = base["papel"].map(grupo_papel)
        base["erro"] = base["y_pred"] - base["y_true"]
        base["abs_error"] = base["erro"].abs()
        base["squared_error"] = base["erro"] ** 2
        print(f"  brutas={len(raw):,} | após {paradigma}={len(pred):,} | avaliadas={len(base):,}")
        bases.append(base); brutas.append(raw)
    return pd.concat(bases, ignore_index=True), pd.concat(brutas, ignore_index=True)

def detectar_lote(df, cfg):
    if cfg.get("col_lote_previsao") and cfg["col_lote_previsao"] in df.columns: return cfg["col_lote_previsao"]
    return next((c for c in cfg["candidatos_col_lote_previsao"] if c in df.columns and df[c].nunique(dropna=True) > 1), None)

In [ ]:
base_erros, previsoes_brutas = construir_base(registra_modelo, CONFIG, PARADIGMA)

metricas_por_serie = base_erros.groupby(["modelo", "grupo_papel", "papel"], observed=True).agg(
    n_pontos=("erro", "count"), MAE=("abs_error", "mean"), MSE=("squared_error", "mean")
).reset_index()

COL_LOTE_USADA = detectar_lote(base_erros, CONFIG)
if COL_LOTE_USADA:
    metricas_por_lote_serie = base_erros.dropna(subset=[COL_LOTE_USADA]).groupby(["modelo", "grupo_papel", "papel", COL_LOTE_USADA], observed=True).agg(
        n_pontos=("erro", "count"), MAE=("abs_error", "mean"), MSE=("squared_error", "mean")
    ).reset_index().rename(columns={COL_LOTE_USADA:"lote_avaliacao"})
    metricas_boxplot = metricas_por_lote_serie.copy()
    print(f"Coluna de lote/janela detectada: {COL_LOTE_USADA}")
else:
    metricas_por_lote_serie = pd.DataFrame()
    metricas_boxplot = metricas_por_serie.copy()
    print("Sem coluna explícita de lote/janela; box plots usarão métricas por série.")

metricas_agregado_grupo = metricas_por_serie.groupby(["modelo", "grupo_papel"], observed=True).agg(n_series=("papel", "nunique"), MAE=("MAE", "mean"), MSE=("MSE", "mean")).reset_index()
metricas_agregado_geral = metricas_por_serie.groupby("modelo", observed=True).agg(n_series=("papel", "nunique"), MAE=("MAE", "mean"), MSE=("MSE", "mean")).reset_index()

display(base_erros.head())
display(metricas_por_serie.sort_values(["grupo_papel", "MAE"]))
display(metricas_agregado_geral.sort_values("MAE"))

In [ ]:
CORES = {r["nome"]: r.get("cor", PALETA_PADRAO[i % len(PALETA_PADRAO)]) for i, r in enumerate(registra_modelo)}

def boxplot(ax, df, metrica, titulo):
    ordem = df.groupby("modelo")[metrica].median().sort_values().index.tolist()
    dados = [df.loc[df["modelo"] == m, metrica].dropna().values for m in ordem]
    bp = ax.boxplot(dados, labels=ordem, patch_artist=True, showfliers=False, medianprops={"linewidth": 2})
    for patch, modelo in zip(bp["boxes"], ordem):
        patch.set_facecolor(CORES.get(modelo))
        patch.set_alpha(0.65)
    ax.set_title(titulo); ax.set_ylabel(metrica); ax.tick_params(axis="x", rotation=35); ax.grid(axis="y", alpha=0.25)

for metrica in METRICAS:
    fig, ax = plt.subplots(figsize=(10, 5))
    boxplot(ax, metricas_boxplot, metrica, f"Agregado — {metrica} — paradigma={PARADIGMA}")
    plt.tight_layout(); plt.show()

for metrica in METRICAS:
    for grupo in sorted(metricas_boxplot["grupo_papel"].dropna().unique()):
        df_g = metricas_boxplot[metricas_boxplot["grupo_papel"] == grupo]
        fig, ax = plt.subplots(figsize=(10, 5))
        boxplot(ax, df_g, metrica, f"Grupo {grupo} — {metrica} — paradigma={PARADIGMA}")
        plt.tight_layout(); plt.show()

In [ ]:
PASTA_SAIDA = Path("outputs/evaluation_05_boxplots")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
base_erros.to_csv(PASTA_SAIDA / f"base_erros_{PARADIGMA}.csv", index=False)
metricas_por_serie.to_csv(PASTA_SAIDA / f"metricas_por_serie_{PARADIGMA}.csv", index=False)
metricas_boxplot.to_csv(PASTA_SAIDA / f"metricas_usadas_boxplot_{PARADIGMA}.csv", index=False)
metricas_agregado_grupo.to_csv(PASTA_SAIDA / f"metricas_agregado_grupo_{PARADIGMA}.csv", index=False)
metricas_agregado_geral.to_csv(PASTA_SAIDA / f"metricas_agregado_geral_{PARADIGMA}.csv", index=False)
if not metricas_por_lote_serie.empty:
    metricas_por_lote_serie.to_csv(PASTA_SAIDA / f"metricas_por_lote_serie_{PARADIGMA}.csv", index=False)
print(f"Arquivos salvos em: {PASTA_SAIDA.resolve()}")